# Autodiff Engine: Gradient Checks and XOR Training

Explore the scalar reverse-mode autodiff engine, verify gradients, and train a tiny MLP on XOR.

In [ ]:
import sys

sys.path.insert(0, "src")

import numpy as np
import math

from from_zero.autodiff import (
    Value,
    topological_sort,
    draw_graph,
    graph_to_dict,
    zero_grad,
    MLP,
    SGD,
    binary_cross_entropy_with_logits,
    mse_loss,
    mean,
    xor_dataset,
)

## 1. Basic Operations and Gradient Accumulation

In [ ]:
# Addition and multiplication
a = Value(3.0)
b = Value(2.0)
c = (a + b) * (a - b) / b**2
print(f"c = {c.data:.4f}")

c.backward()
print(f"a.grad = {a.grad:.4f}")
print(f"b.grad = {b.grad:.4f}")

# Shared subexpression: gradients must accumulate
x = Value(2.0)
y = Value(3.0)
shared = x * y
out = shared + shared
out.backward()
print(f"Shared subexpression:")
print(f"  x.grad = {x.grad:.4f} (expected 6.0)")
print(f"  y.grad = {y.grad:.4f} (expected 4.0)")

## 2. Topological Sort and Graph Visualization

In [ ]:
# Build a small expression graph
a = Value(1.0)
b = Value(2.0)
c = a * b
d = c + a
e = d.tanh()

order = topological_sort(e)
print(f"Graph has {len(order)} nodes")
for node in order:
    parents = [id(p) for p in node.parents]
    print(f"  {id(node)}: {node.op} data={node.data:.4f} parents={parents}")

# Visualize
fig = draw_graph(e, max_nodes=50)
import matplotlib.pyplot as plt

plt.show()

## 3. Numerical Gradient Check

Compare autodiff gradients with central finite differences on a random expression.

In [ ]:
def autodiff_fn(vals):
    a, b, c = [Value(float(v)) for v in vals]
    f1 = (a * b + c).tanh()
    f2 = (a**2 + 0.5 * b).sigmoid()
    f3 = (c.exp() + 1.0).log()
    return f1 + f2 + f3 + (a / (b + 2.0)) - 0.25 * c**2


point = np.array([0.7, 1.3, -0.4])
root = autodiff_fn(point)
root.backward()

# Extract leaf gradients (need to rebuild to get fresh leaves)
a, b, c = [Value(float(v)) for v in point]
f1 = (a * b + c).tanh()
f2 = (a**2 + 0.5 * b).sigmoid()
f3 = (c.exp() + 1.0).log()
out = f1 + f2 + f3 + (a / (b + 2.0)) - 0.25 * c**2
out.backward()
analytic = np.array([a.grad, b.grad, c.grad])


# Central differences
def central_diff(fn, x, h=1e-6):
    grad = np.zeros_like(x)
    for i in range(len(x)):
        x_plus = x.copy()
        x_plus[i] += h
        x_minus = x.copy()
        x_minus[i] -= h
        grad[i] = (fn(x_plus).data - fn(x_minus).data) / (2 * h)
    return grad


numeric = central_diff(autodiff_fn, point)
print(f"Analytic: {analytic}")
print(f"Numeric:  {numeric}")
print(f"Max abs error: {np.max(np.abs(analytic - numeric)):.2e}")

## 4. MLP on XOR

In [ ]:
# Build model: 2 -> 4 -> 4 -> 1
model = MLP(2, [4, 4], 1, activation="tanh", output_activation="linear", seed=42)
print(f"Parameter count: {model.parameter_count()}")

# XOR dataset
data = xor_dataset()
print(f"Dataset: {len(data)} samples")
for xi, yi in data:
    print(f"  {xi} -> {yi}")

In [ ]:
# Training loop
optimizer = SGD(model.parameters(), learning_rate=0.2)
losses = []

for epoch in range(5000):
    epoch_loss = 0.0
    for xi, yi in data:
        x_vals = [Value(v) for v in xi]
        y_pred = model(x_vals)
        loss = mse_loss([y_pred], [Value(float(yi))])
        model.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.data
    losses.append(epoch_loss / len(data))

    if epoch % 500 == 0:
        print(f"Epoch {epoch}: loss = {losses[-1]:.6f}")

# Final evaluation
correct = 0
for xi, yi in data:
    x_vals = [Value(v) for v in xi]
    y_pred = model(x_vals)
    pred = 1 if y_pred.data > 0.5 else 0
    correct += pred == yi

print(f"\nFinal loss: {losses[-1]:.6f}")
print(f"Accuracy: {correct}/4 = {correct / 4:.2f}")
print(f"Solved: {correct == 4}")

In [ ]:
# Plot loss curve
plt.plot(losses)
plt.yscale("log")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss (log)")
plt.title("XOR Training Loss")
plt.grid(alpha=0.3)
plt.show()

## 5. Decision Boundary Visualization

In [ ]:
# Create grid
xx, yy = np.meshgrid(np.linspace(-0.5, 1.5, 50), np.linspace(-0.5, 1.5, 50))
grid_points = np.c_[xx.ravel(), yy.ravel()]

preds = []
for pt in grid_points:
    x_vals = [Value(v) for v in pt]
    y_pred = model(x_vals)
    preds.append(y_pred.data)

preds = np.array(preds).reshape(xx.shape)

plt.contourf(xx, yy, preds, levels=20, alpha=0.6, cmap="RdBu")
plt.colorbar(label="Predicted value")

# Plot XOR points
for xi, yi in data:
    color = "red" if yi == 0 else "blue"
    plt.scatter(xi[0], xi[1], c=color, s=100, edgecolor="black", zorder=5)

plt.xlabel("x1")
plt.ylabel("x2")
plt.title("MLP Decision Boundary on XOR")
plt.show()